In [1]:
# hybrid_rating_pipeline_complete.py
# Pipeline LLM + Grover-like Oracle + evaluation (RMSE, MAE)
# Requirements:
# pip install pennylane transformers torch pandas scikit-learn

import math, re, sys, json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

import pennylane as qml
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


/faststorage/project/DEIC-SDU-L2-22/env/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))
/faststorage/project/DEIC-SDU-L2-22/env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ----------------------------
# Fix random seeds for reproducibility
# ----------------------------
np.random.seed(42)
torch.manual_seed(42)


# ----------------------------
# Config
# ----------------------------
MIN_RATING = 0
MAX_RATING = 5
NUM_RATINGS = MAX_RATING - MIN_RATING + 1  # e.g., 6
# default small LLM for local tests
LLM_NAME = "distilgpt2"
# PennyLane device shots
QL_SHOTS = 512

In [3]:
# ----------------------------
# Utilities
# ----------------------------
def n_qubits_for_num_ratings(num_ratings):
    return math.ceil(math.log2(num_ratings))

N_QUBITS = n_qubits_for_num_ratings(NUM_RATINGS)
print(f"Using {N_QUBITS} qubits (can represent {2**N_QUBITS} states) for {NUM_RATINGS} ratings")

def rating_to_index(r):
    """Map rating in [MIN_RATING,MAX_RATING] to index 0..NUM_RATINGS-1"""
    idx = int(round(float(r))) - MIN_RATING
    idx = max(0, min(NUM_RATINGS - 1, idx))
    return idx

def index_to_rating(idx):
    return MIN_RATING + int(idx)

def index_to_bits(idx, n_qubits=N_QUBITS):
    b = format(int(idx), f"0{n_qubits}b")
    return [int(ch) for ch in b]

def bits_to_index(bits):
    return int("".join(str(b) for b in bits), 2)

Using 3 qubits (can represent 8 states) for 6 ratings


In [5]:
# ----------------------------
# LLM helper
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model = AutoModelForCausalLM.from_pretrained(LLM_NAME)
# If model doesn't have pad_token, avoid warnings by setting
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

def make_prompt(row):
    # Keep prompt explicit and directive to get a single number
    return (
        f"User: sex={row.get('sex','?')}, age={row.get('age','?')}, country={row.get('Country','?')}, mood={row.get('mood','?')}\n"
        f"Movie: title={row.get('Movie_Name','?')}, director={row.get('Director','?')}, genres={row.get('Genre1','?')}/{row.get('Genre2','?')}\n"
        "Question: On a scale from 0 to 5 (inclusive), what rating would the user give this movie? "
        "Answer with only a single number between 0 and 5 (e.g., 3 or 4.5).\nAnswer: "
    )

# def extract_rating_from_text(text):
#     # find the first occurrence of a number between 0 and 5 (allow decimal)
#     m = re.search(r'(?<!\d)([0-5](?:\.\d+)?)', text)
#     if m:
#         val = float(m.group(1))
#         return max(MIN_RATING, min(MAX_RATING, val))
#     # fallback: return midpoint
#     return float((MIN_RATING + MAX_RATING) / 2.0)

# def llm_predict_row(row, max_new_tokens=20):
#     prompt = make_prompt(row)
#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
#     out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
#     decoded = tokenizer.decode(out[0], skip_special_tokens=True)
#     # sometimes model repeats prompt; try to take the trailing part after "Answer"
#     if "Answer" in decoded:
#         decoded = decoded.split("Answer", 1)[1]
#     rating = extract_rating_from_text(decoded)
#     return rating, decoded.strip()
def extract_rating_from_text(text):
    # Try to find the last "Answer" block then search for number there (safer)
    # Normalize separators
    text = text.replace("\n", " ").strip()
    # If there is 'Answer' or 'Answer:' take the substring after the last occurrence
    if "Answer" in text:
        parts = re.split(r'Answer[:\s]*', text, flags=re.IGNORECASE)
        candidate = parts[-1] if parts[-1].strip() != "" else text
    else:
        candidate = text
    # Now search for a number between 0 and 5 (allow decimals)
    m = re.search(r'(?<!\d)([0-5](?:\.\d+)?)(?!\d)', candidate)
    if m:
        val = float(m.group(1))
        return max(MIN_RATING, min(MAX_RATING, val))
    # fallback: search the whole text (last chance)
    m2 = re.search(r'([0-5](?:\.\d+)?)', text)
    if m2:
        return max(MIN_RATING, min(MAX_RATING, float(m2.group(1))))
    # final fallback: midpoint
    return float((MIN_RATING + MAX_RATING) / 2.0)

def llm_predict_row(row, max_new_tokens=20):
    prompt = make_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    # Ensure model and inputs on same device
    device = next(model.parameters()).device
    inputs = {k:v.to(device) for k,v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # debug: return decoded for inspection
    rating = extract_rating_from_text(decoded)
    return rating, decoded.strip()

In [6]:
# Calcul automatique du nombre d’itérations Grover, j'ai ajouté une fonction utilitaire pour calculer k optimal :
def optimal_grover_iterations(n_qubits, n_marked=1):
    N = 2 ** n_qubits
    M = max(1, n_marked)
    k = int(math.floor((math.pi / 4.0) * math.sqrt(N / M)))
    return max(1, k)

In [7]:
def safe_multi_controlled_x(controls, target):
    # Try to call the native op; if not available, implement a simple decomposition for small n
    try:
        qml.MultiControlledX(wires=controls + [target])
    except Exception as e:
        # Decomposition for 1 or 2 controls (toy fallback)
        if len(controls) == 1:
            qml.CNOT(wires=[controls[0], target])
        elif len(controls) == 2:
            # Toffoli
            qml.Toffoli(wires=[controls[0], controls[1], target])
        else:
            raise RuntimeError("MultiControlledX not available and no fallback implemented for >2 controls.")


In [8]:
# ----------------------------
# Quantum: Oracle & Diffusion (Grover-like)
# ----------------------------
def apply_oracle_ops(target_bits):
    # returns a python function that applies the oracle marking target_bits with a phase flip
    def oracle():
        # flip qubits where bit == 0
        for i, b in enumerate(target_bits):
            if b == 0:
                qml.PauliX(wires=i)
        # multi-controlled Z via H on target & MultiControlledX
        if len(target_bits) == 1:
            qml.PauliZ(wires=0)
        else:
            target = len(target_bits) - 1
            qml.Hadamard(wires=target)
            controls = list(range(0, len(target_bits) - 1))
            safe_multi_controlled_x(controls, target)
            qml.Hadamard(wires=target)
        # undo flips
        for i, b in enumerate(target_bits):
            if b == 0:
                qml.PauliX(wires=i)
    return oracle

In [9]:
def diffusion_ops(n_qubits):
    def diffuse():
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
        for i in range(n_qubits):
            qml.PauliX(wires=i)
        if n_qubits == 1:
            qml.PauliZ(wires=0)
        else:
            target = n_qubits - 1
            qml.Hadamard(wires=target)
            controls = list(range(0, n_qubits - 1))
            safe_multi_controlled_x(controls, target)
            qml.Hadamard(wires=target)
        for i in range(n_qubits):
            qml.PauliX(wires=i)
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
    return diffuse

In [10]:
def build_grover_qnode(n_qubits, shots=QL_SHOTS, interface='autograd'):
    dev = qml.device("default.qubit", wires=n_qubits, shots=shots)
    def make_qnode(k_iter):
        @qml.qnode(dev, interface=interface)
        def qnode(target_bits):
            # prepare uniform superposition
            for i in range(n_qubits):
                qml.Hadamard(wires=i)
            oracle_fn = apply_oracle_ops(target_bits)
            diffuse_fn = diffusion_ops(n_qubits)
            for _ in range(k_iter):
                oracle_fn()
                diffuse_fn()
            # sample computational basis on all wires
            return qml.sample(wires=list(range(n_qubits)))
        return qnode
    return make_qnode


In [11]:
# ----------------------------
# Run Grover and decode predicted rating (most frequent bitstring)
# ----------------------------
# def run_grover_get_rating(qnode, target_bits, k_iter=1):
#     # qnode returns array shape (shots, n_qubits) typically
#     samples = qnode(target_bits)  # may be numpy array
#     # Convert samples to bitstrings
#     samples_arr = np.array(samples)
#     # If shape is (n_qubits, shots) (older API), transpose
#     if samples_arr.ndim == 2 and samples_arr.shape[0] == N_QUBITS and samples_arr.shape[1] == qnode.device.num_shots:
#         samples_arr = samples_arr.T
#     # Now samples_arr is (shots, n_qubits) with values 0/1 typically
#     bitstrings = ["".join(str(int(b)) for b in row) for row in samples_arr]
#     counts = Counter(bitstrings)
#     top_bits, top_count = counts.most_common(1)[0]
#     predicted_index = bits_to_index([int(ch) for ch in top_bits])
#     # if predicted index maps beyond NUM_RATINGS (because 2^n > NUM_RATINGS), clamp:
#     predicted_index = min(predicted_index, NUM_RATINGS - 1)
#     predicted_rating = index_to_rating(predicted_index)
#     return predicted_rating, counts
# je l'ai améliorer en ajoutant  aussi la distribution counts triée et la probabilité du top
def run_grover_get_rating(qnode, target_bits, k_iter=1):
    samples = qnode(target_bits)
    samples_arr = np.array(samples)
    # handle shape variants
    if samples_arr.ndim == 2 and samples_arr.shape[0] == N_QUBITS and samples_arr.shape[1] == qnode.device.shots:
        samples_arr = samples_arr.T
    if samples_arr.ndim == 1:
        # single sample case -> expand
        samples_arr = np.expand_dims(samples_arr, axis=0)
    bitstrings = ["".join(str(int(b)) for b in row) for row in samples_arr]
    counts = Counter(bitstrings)
    total = sum(counts.values())
    top_bits, top_count = counts.most_common(1)[0]
    predicted_index = bits_to_index([int(ch) for ch in top_bits])
    predicted_index = min(predicted_index, NUM_RATINGS - 1)
    predicted_rating = index_to_rating(predicted_index)
    # also produce normalized distribution dictionary
    distr = {k: v / total for k, v in counts.items()}
    return predicted_rating, counts, distr


In [12]:
# ----------------------------
# Helper: choose marked targets around llm_pred (±radius)
# ----------------------------
def make_marked_indices(llm_pred, radius=0):
    """Return list of indices to mark. radius=0 -> only rounded(llm_pred).
       radius=1 -> round(llm_pred) ± 1 within bounds, etc."""
    center = rating_to_index(llm_pred)
    idxs = []
    for d in range(-radius, radius + 1):
        idx = center + d
        if 0 <= idx < NUM_RATINGS:
            idxs.append(idx)
    # ensure unique
    return sorted(set(idxs))


In [10]:
# ----------------------------
# Hybrid prediction per row
# ----------------------------
def hybrid_predict_row(row, qnode_maker, grover_iters=1, llm_weight=0.6, radius_mark=0):
    llm_pred, llm_text = llm_predict_row(row)
    # form marked set
    marked_idxs = make_marked_indices(llm_pred, radius=radius_mark)
    # we will mark all these indices in the oracle (multi-target marking by marking each target in sequence)
    # Implementation strategy: in the qnode we call oracle corresponding to each target sequentially before diffusion,
    # but easier: run Grover separately for each marked idx and pick most confident? To keep simple & stable:
    # We'll mark only the rounded prediction (radius=0) OR if radius>0 we'll run Grover with each target and average.
    if len(marked_idxs) == 1:
        target_idx = marked_idxs[0]
        target_bits = index_to_bits(target_idx, n_qubits=N_QUBITS)
         # 🔹 Choisir automatiquement le nombre optimal d’itérations
        optimal_iters = optimal_grover_iterations(N_QUBITS, len(marked_idxs))
        qnode = qnode_maker(optimal_iters)
        q_pred, counts = run_grover_get_rating(qnode, target_bits, k_iter=optimal_iters)
        # qnode = qnode_maker(grover_iters)
        # q_pred, counts = run_grover_get_rating(qnode, target_bits, k_iter=grover_iters)
    else:
        # run Grover for each marked index and choose the most frequently returned rating
        votes = Counter()
        counts_across = {}
        for idx in marked_idxs:
            bits = index_to_bits(idx, n_qubits=N_QUBITS)
            qnode = qnode_maker(grover_iters)
            q_pred_tmp, counts_tmp = run_grover_get_rating(qnode, bits, k_iter=grover_iters)
            votes[q_pred_tmp] += 1
            counts_across[idx] = counts_tmp
        q_pred = votes.most_common(1)[0][0]
        counts = counts_across
    # combine
    final = llm_weight * float(llm_pred) + (1.0 - llm_weight) * float(q_pred)
    return {
        "llm_pred": float(llm_pred),
        "llm_text": llm_text,
        "q_pred": float(q_pred),
        "counts": counts,
        "final": float(final)
    }


In [11]:

# ----------------------------
# Full evaluation on dataset (RMSE / MAE)
# ----------------------------
def evaluate_on_dataframe(df, label_col="rating", test_size=0.2, random_state=42,
                          grover_iters=1, llm_weight=0.6, radius_mark=0, max_rows=None):
    # optional: limit rows for quick tests
    if max_rows:
        df = df.iloc[:max_rows].copy()
    # basic cleaning: ensure label exists
    df = df.dropna(subset=[label_col])
    train_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state)
    print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
    qnode_maker = build_grover_qnode(N_QUBITS, shots=QL_SHOTS)
    preds = []
    trues = []
    for i, row in test_df.iterrows():
        res = hybrid_predict_row(row, qnode_maker, grover_iters=grover_iters, llm_weight=llm_weight, radius_mark=radius_mark)
        preds.append(res['final'])
        trues.append(float(row[label_col]))
        if i % 10 == 0:
            print(f"Idx {i}: llm {res['llm_pred']:.3f}, q {res['q_pred']:.1f}, final {res['final']:.3f}, true {row[label_col]}")
    rmse = math.sqrt(mean_squared_error(trues, preds))
    mae = mean_absolute_error(trues, preds)
    return {"rmse": rmse, "mae": mae, "n": len(trues)}


In [15]:
!pip install openpyxl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     |████████████████████████████████| 250 kB 8.2 MB/s            
You should consider upgrading via the '/faststorage/project/DEIC-SDU-L2-22/env/bin/python3 -m pip install --upgrade pip' command.


In [17]:
# ----------------------------
# Example usage
# ----------------------------
if __name__ == "__main__":
    # Path to your LDOS CSV file (adapt)
    csv_path = "datasetLDOS.xlsx"  # <--- change to your real path
    try:
        df = pd.read_excel(csv_path)
    except Exception as e:
        print(f"Error reading {csv_path}: {e}")
        print("Creating a small toy DataFrame for demo...")
        df = pd.DataFrame([
            {"userID":26, "sex":"Female", "age":26, "Country":"United Kingdom", "mood":"Neutral",
             "Movie_Name":"Priest", "Director":"Antonia Bird", "Genre1":"Drama", "rating":2},
            {"userID":26, "sex":"Female", "age":26, "Country":"United States", "mood":"Neutral",
             "Movie_Name":"Geek Charming", "Director":"K. Asher Levin", "Genre1":"Comedy", "rating":3},
            {"userID":27, "sex":"Male", "age":35, "Country":"France", "mood":"Happy",
             "Movie_Name":"Random Film", "Director":"Someone", "Genre1":"Action", "rating":4},
        ])

    # Quick evaluate (set max_rows small if you want fast execution)
    metrics = evaluate_on_dataframe(df, label_col="rating", test_size=0.5,
                                    grover_iters=1, llm_weight=0.6, radius_mark=0, max_rows=10)
    print("Evaluation:", metrics)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Train size: 5, Test size: 5


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Idx 0: llm 0.000, q 0.0, final 0.000, true 2
Evaluation: {'rmse': 3.492849839314596, 'mae': 3.4, 'n': 5}
